In [ ]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor 
from langchain.prompts import ChatPromptTemplate

import utils

# Create model
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# Mock API Tool: Check Claim Status
@tool
def check_claim_status(claim_id: str) -> str:
    """Return status of a claim with given claim_id."""
    fake_db = {
        "C101": "Approved - Payment in process",
        "C202": "Under review - Waiting documents",
        "C303": "Flagged for manual investigation"
    }
    return fake_db.get(claim_id, "Claim ID not found")

# Fraud Detection Tool
@tool
def fraud_risk_check(description: str) -> str:
    """Returns risk evaluation for fraud based on description."""
    if "lost phone but still using it" in description.lower():
        return "High Risk"
    return "Low Risk"

# Escalation Tool
@tool
def escalate_to_human(issue: str) -> str:
    """Escalates conversation to a human agent."""
    return f"Escalation created. A human agent will review: {issue}"

In [ ]:
tools = [check_claim_status, fraud_risk_check, escalate_to_human]

SYSTEM_PROMPT = """
You are an Insurance Claims Assistant.

Your job:
1) Understand user intent.
2) If they ask about a claim status → call `check_claim_status`
3) If user mentions incident details → call `fraud_risk_check`
4) If unsure, conflicting, or user upset → call `escalate_to_human`

Always respond in a friendly, formal tone.
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [ ]:
from openai import max_retries


agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)

In [ ]:
query = "What's the current status of claim C202?"
result = executor.invoke({"input": query})
print("Agent Output:", result)

In [ ]:
query = "I lost my phone but it looks like it is being used by someone."
result = executor.invoke({"input": query})
print("Agent Output:", result)